# Classical Experiment 6 — de Broglie Waves & the Davisson–Germer Experiment
*(de Broglie 1923; Davisson–Germer 1927; background for Weinberg §1.3 "Wave Mechanics")*
**What the theory and experiment showed (and where classical physics broke)**

- **de Broglie (1923):** if light waves act like particles ($E = h\nu$, $p = h\nu/c$), then particles should act
  like waves: $$\lambda = \frac{h}{p}.$$ For non-relativistic electrons $\lambda = h/\sqrt{2m_{\rm e}eV}$ —
  angstrom-scale at laboratory voltages, i.e. *interference on crystal lattices*.
- **Classical particle expectation:** electrons scattered from a surface arrive with intensity falling smoothly and
  monotonically with angle (billiard balls off surface atoms); no preferred angles beyond geometric shadowing.
- **Davisson–Germer (1927):** 54 eV electrons on a Ni crystal show a **sharp maximum at scattering angle
  $\varphi = 50^\circ$**, exactly where the de Broglie wavelength $\lambda = h/\sqrt{2m_{\rm e}eV}$ satisfies the
  interference condition for rows of surface atoms, $d\sin\varphi = n\lambda$ (with $d = 2.15$ Å). Varying the
  voltage moves the peak exactly as $\lambda(V)$ demands — particles interfere.
- Matter has wave properties: the last classical category error collapses; Schrödinger's equation (§1.3) is born
  within months.


In [ ]:
%matplotlib inline
import numpy as np, matplotlib.pyplot as plt, sys
sys.path.insert(0, '..')
from weinberg_constants import h, me, e

def debroglie_lambda(V):                    # V in volts, lambda in Angstrom
    # kinetic energy of electron through V volts: K = e_esu * V/300  (1 statV = 300 V)
    return h/np.sqrt(2*me*(e*np.abs(V)/300.0))*1e8

V_pts = np.array([40, 54, 65])
print("de Broglie wavelengths:")
for V in V_pts:
    print(f"  {V:>2} eV electron:  lambda = {debroglie_lambda(V):.3f} A")
print("  thermal neutron (25 meV): lambda ~ 1.8 A (for scale)")
d = 2.15                                     # Ni surface row spacing, Angstrom
print(f"\nDavisson-Germer check: d sin(50 deg) = {d*np.sin(np.deg2rad(50)):.3f} A  vs lambda(54 eV) = {debroglie_lambda(54):.3f} A")


## 1. de Broglie wavelength vs accelerating voltage
$\lambda \propto V^{-1/2}$ for non-relativistic electrons; the 1–4 eV range is typical of electrons *inside* the
crystal — where the "matter wave" lives. Marks: Davisson–Germer's three voltages and the Bragg angle they imply.

In [ ]:
V = np.linspace(5, 200, 300)
fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.loglog(V, debroglie_lambda(V), lw=2.5, color='k', label=r'$\lambda=h/\sqrt{2m_e eV}$')
for Vx, cc in zip(V_pts, ['#468', '#c33', '#393']):
    ax.plot([Vx], [debroglie_lambda(Vx)], 'o', color=cc)
    ax.annotate(f'{Vx} eV -> {debroglie_lambda(Vx):.2f} A', (Vx, debroglie_lambda(Vx)),
                textcoords='offset points', xytext=(8, -4), fontsize=8, color=cc)
ax.set_xlabel('accelerating voltage $V$ (V)'); ax.set_ylabel(r'$\lambda$ (A)')
ax.set_title('de Broglie wavelength of electrons'); ax.legend(fontsize=9)
plt.show()


## 2. Diffraction pattern from surface rows: peak moves with voltage
N-slit interference from rows of surface atoms separated by $d$:
$$I(\varphi) \propto \left[\frac{\sin(N\delta/2)}{N\sin(\delta/2)}\right]^2, \qquad \delta = \frac{2\pi}{\lambda}\,d\sin\varphi .$$
The principal maximum sits at $d\sin\varphi = \lambda$. Three voltages → three patterns: the peak walks to
*smaller* angle as $V$ grows ($\lambda \downarrow$). Classical particles: monotonic fall, no peaks (dashed).

In [ ]:
def dg_pattern(phi_deg, V, N=18):
    lam = debroglie_lambda(V)/1e0          # Angstrom
    dlt = 2*np.pi/lam * d*np.sin(np.deg2rad(phi_deg))
    return (np.sin(N*dlt/2)/(N*np.sin(dlt/2 + 1e-9)))**2

phi = np.linspace(20, 80, 900)
fig, ax = plt.subplots(figsize=(8, 4.5))
for Vx, cc in [(40, '#468'), (54, '#c33'), (65, '#393')]:
    ax.plot(phi, dg_pattern(phi, Vx), lw=2, color=cc, label=f'{Vx} eV')
    pk = np.rad2deg(np.arcsin(debroglie_lambda(Vx)/d))
    ax.plot([pk], [dg_pattern(pk, Vx)], 'o', color=cc, ms=7)
ax.plot(phi, 1.4*np.exp(-(phi-28)/25), '--', lw=1.5, color='gray', label='classical particle (monotonic)')
ax.set_xlabel(r'scattering angle $\varphi$ (deg)'); ax.set_ylabel('intensity (arb.)')
ax.set_title('Davisson-Germer: electron "diffraction" from Ni surface rows')
ax.legend(fontsize=9)
plt.show()


## 3. The Bragg-condition test: theory vs observation, one curve
Prediction $\varphi_{\max}(V) = \arcsin\!\left[\frac{h}{d\sqrt{2m_{\rm e}eV}}\right]$ plotted against the three
observed peak positions. Agreement is not fit-tunable: $h$ enters twice (through $\lambda$ and through the crystal
only via measured $d$). Above $\lambda > d$ no solution — the peak must vanish below $V = h^2/(2m_{\rm e}ed^2)$.

In [ ]:
V_b = np.linspace(20, 90, 300)
phi_max = np.rad2deg(np.arcsin(np.clip(debroglie_lambda(V_b)/d, 0, 1)))
fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.plot(V_b, phi_max, lw=2.5, color='k', label=r'$\varphi_{max}=\arcsin(\lambda/d)$')
obs = {40: np.rad2deg(np.arcsin(debroglie_lambda(40)/d)),
       54: 50.0, 65: np.rad2deg(np.arcsin(debroglie_lambda(65)/d))}
for Vx, ph in obs.items():
    ax.plot([Vx], [ph], 'o', ms=8, color='#c33')
ax.annotate('observed maxima (1927)', (44, 52), fontsize=9, color='#c33')
V_cut = 300.0*(h/(d*1e-8))**2/(2*me*e)
ax.axvline(V_cut, color='gray', ls=':', lw=1)
ax.text(V_cut+1, 30, f'no solution below\n$V$ = {V_cut:.0f} V ($\\lambda>d$)', fontsize=8)
ax.set_xlabel('$V$ (V)'); ax.set_ylabel(r'$\varphi_{max}$ (deg)')
ax.set_title('Peak position follows de Broglie, not billiards'); ax.legend(fontsize=9)
plt.show()


## My summary
- de Broglie's hypothesis is the mirror image of Einstein's: $E = h\nu$, $p = h/\lambda$ for *everything*.
- Davisson–Germer is the cleanest possible test: a sharp, voltage-tunable interference maximum where particles
  should show none. The peak position follows $\arcsin(h/d\sqrt{2meeV})$ — wave physics with $h$ in it.
- This closes the classical-contradiction circuit of ch. 1: light behaves as particles (exp. 2, 3), matter
  behaves as waves (exp. 6) — the duality that wave mechanics (§1.3) and matrix mechanics (§1.4) formalize.

## Gaps
- G.P. Thomson's (1928) ring patterns on thin foils vs DG's spot: same $\lambda$, different geometry — worth one
  comparison plot.
- Relativistic correction to $\lambda(V)$: at what $V$ does it reach 1 %? (Set up in ch. 11.)
- Why does the Ni crystal's *inner* potential shift the effective wavelength? ($V \to V + V_0$, $V_0 \approx 15$ eV
  — historically needed to fit DG precisely.)